# demo_multi

> Four circuit agents sharing one arena through the Phase 5 event queue.

Spec: `specs/multiagent/phase5/` (design-low-level section 11). Artifacts under `results/multiagent/demo_multi/` (gitignored). Agents do not interact in Phase 5, so each agent's events must equal its run alone. Normal tests use 300 s; `#| slow` runs 1200 s (`full/`).

In [ ]:
from explRL.circuit import goalStats
from explRL.core import loadCircuitAgent
from explRL.envs import ZoneArena
from explRL.members import CircuitMember
from explRL.multi import runTeam
from explRL.circuit_train import historyFromRecords, goalSafeDistance, piScore, heldValue
from explRL.circuit_runs import goalDistanceNow, lastQuarterMean
from explRL.train import demo_artifact_dir, save_demo_figure, save_demo_metrics
import matplotlib.pyplot as plt
import numpy as np

DN = 'multiagent/demo_multi'
artifact_mode, Ttot = 'quick', 300.0
seed, nAgents = 0, 4

In [ ]:
#| slow
artifact_mode, Ttot = 'full', 1200.0

In [ ]:
env = ZoneArena(symmetric=True)
members = [CircuitMember(loadCircuitAgent(seed=seed + i)) for i in range(nAgents)]
starts = [dict(dist=goalDistanceNow(m.agent, env),
               strength=goalStats(m.agent.policy.wG)['strength']) for m in members]
team = runTeam(members, env, Ttot, t0=0.0)
tS = members[0].agent.params.tS

## Event timeline

Each dot is one iteration (saccade, possible jump, fixation) starting at that time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
for i in range(nAgents):
    ts = [r.t for r in team.forAgent(i)]
    ax.scatter(ts, [i] * len(ts), s=8)
ax.set_yticks(range(nAgents)); ax.set_xlabel('time (s)'); ax.set_ylabel('agent'); ax.set_title('Team events')
plt.tight_layout()
save_demo_figure('timeline', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()

## Learning dynamics

Each line is one agent. **PI** is fixation-weighted safe minus danger time (higher is better). **Distance** is degrees from the learned goal in the arena to the nearest safe-zone center. **Jump rate** is the running fraction of saccades followed by a 180° compass jump (exploratory remapping in the symmetric arena). Agents do not interact; curves differ only by seed.

In [ ]:
def _iterEnd(h):
    return h.t + h.fixDur + tS

def _cumPI(h):
    w = h.valence * h.fixDur
    return _iterEnd(h), np.cumsum(w) / np.cumsum(h.fixDur)

def _cumJump(h):
    n = np.arange(1, len(h.t) + 1, dtype=float)
    return _iterEnd(h), np.cumsum(h.jumped) / n

times = np.arange(0.0, Ttot, 1.0)
histories = [historyFromRecords(team.forAgent(i)) for i in range(nAgents)]
learning = []
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for i, h in enumerate(histories):
    dist = goalSafeDistance(h, env)
    s0 = starts[i]
    learning.append(dict(agent=i, dist_start=s0['dist'], dist_end=float(dist[-1]),
                         dist_last_quarter=lastQuarterMean(h, dist, s0['dist'], Ttot, tS),
                         strength_start=s0['strength'], strength_end=float(h.strength[-1]),
                         PI=piScore(h), jump_rate=float(h.jumped.mean())))
    axes[0].plot(*_cumPI(h), label=f'agent {i}')
    axes[1].plot(times, heldValue(h, dist, s0['dist'], times, tS), label=f'agent {i}')
    axes[2].plot(*_cumJump(h), label=f'agent {i}')
for ax, ylab in zip(axes, ('PI (cumulative)', 'goal distance to safe (deg)', 'jump rate (cumulative)')):
    ax.set_ylabel(ylab); ax.legend(loc='best', fontsize=8)
axes[-1].set_xlabel('time (s)')
plt.tight_layout()
save_demo_figure('learning', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()
learning

## Independence check (AC-2)

In [ ]:
def sameAsSolo(i):
    solo = runTeam([CircuitMember(loadCircuitAgent(seed=seed + i))], env, Ttot, t0=0.0).records
    mine = team.forAgent(i)
    return len(mine) == len(solo) and all(a._replace(agent=0) == b for a, b in zip(mine, solo))

independent = all(sameAsSolo(i) for i in range(nAgents))
print('each agent equals its solo run:', independent)
assert independent

## JSONL export and metrics

In [ ]:
path = team.toJSONL(demo_artifact_dir(artifact_mode, name=DN) / 'team.jsonl')
print(''.join(path.read_text().splitlines(True)[:3]))
counts = [len(team.forAgent(i)) for i in range(nAgents)]
save_demo_metrics(dict(mode=artifact_mode, seed=seed, nAgents=nAgents, Ttot_s=Ttot,
                       scene_symmetric=env.symmetric, events_per_agent=counts,
                       independent=independent, learning_outcomes=learning),
                      mode=artifact_mode, demo_name=DN)
counts